# Amazon ML Challenge 2026: Business Entity Resolution
**Team: PERCEPTRON**

This notebook automates the complete workflow on **Google Colab**:
1. Mount Google Drive and automatically extract `6ab10eb3b23ba_student_resource.zip`.
2. Clone the private repository `amazon-ml-challenge-2026`.
3. Install dependencies.
4. Execute the multi-stage Entity Resolution pipeline (Blocking + Pairwise Features + LightGBM + F0.5 Calibration).
5. Run the official validator `utils/validate_submission.py`.
6. Package `PERCEPTRON_submission.zip` and copy it directly back to your Google Drive!

--- 
## Step 1: Mount Google Drive
Make sure you uploaded `6ab10eb3b23ba_student_resource.zip` into your Google Drive (any folder or the root `MyDrive`).

In [ ]:
from google.colab import drive
import os, sys, shutil, zipfile
from pathlib import Path

drive.mount('/content/drive')

--- 
## Step 2: Clone Private GitHub Repository
Enter your GitHub Personal Access Token (PAT) when prompted.
*(Create one in 10 seconds: GitHub -> Settings -> Developer Settings -> Personal access tokens -> Tokens (classic) -> Check 'repo' checkbox)*.

In [ ]:
import getpass

gh_token = getpass.getpass('Enter your GitHub Personal Access Token: ')

!rm -rf /content/amazon-ml-challenge-2026
!git clone https://{gh_token}@github.com/SabarishR08/amazon-ml-challenge-2026.git /content/amazon-ml-challenge-2026
%cd /content/amazon-ml-challenge-2026
!pip install -r code/business_entity_resolution/requirements.txt

--- 
## Step 3: Locate and Extract Student Resource Zip from Google Drive
This finds the zip file in your Google Drive and extracts it directly onto Colab's high-speed local SSD.

In [ ]:
zip_name = '6ab10eb3b23ba_student_resource.zip'
zip_candidates = list(Path('/content/drive/MyDrive').rglob(zip_name))

if not zip_candidates:
    zip_candidates = list(Path('/content/drive/MyDrive').rglob('*student_resource*.zip'))

if zip_candidates:
    zip_path = zip_candidates[0]
    print(f'Found dataset zip at: {zip_path}')
    print('Extracting directly into /content/amazon-ml-challenge-2026...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        for item in z.infolist():
            if item.filename.startswith('__MACOSX') or item.filename.endswith('.DS_Store'):
                continue
            rel = item.filename
            if rel.startswith('student_resource/'):
                rel = rel[len('student_resource/'):]
            if not rel:
                continue
            target = os.path.join(os.getcwd(), rel)
            if item.is_dir():
                os.makedirs(target, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with z.open(item) as src, open(target, 'wb') as dst:
                    dst.write(src.read())
    print('Extraction complete!')
else:
    print('ERROR: Zip file not found in Google Drive! Please verify file name in MyDrive.')

--- 
## Step 4: Verify Extracted TSV Files

In [ ]:
!ls -lh dataset/train
!ls -lh dataset/test

--- 
## Step 5: Execute Complete Entity Resolution Pipeline
Runs candidate generation (blocking), pairwise feature calculation, and LightGBM classification with precision threshold calibration ($t = 0.68$).

In [ ]:
!python code/business_entity_resolution/src/pipeline.py \
    --train-dir dataset/train \
    --test-dir dataset/test \
    --output-dir output \
    --threshold 0.68

--- 
## Step 6: Validate and Package Final Submission
Validates `matching_results.tsv` and `candidate_pairs.tsv` with the official rules and saves `PERCEPTRON_submission.zip` straight into your Google Drive.

In [ ]:
!python package_submission.py --team-name PERCEPTRON

# Copy to Google Drive
drive_output_path = '/content/drive/MyDrive/PERCEPTRON_submission.zip'
shutil.copy('PERCEPTRON_submission.zip', drive_output_path)
print(f'Successfully copied final submission zip to: {drive_output_path}')